In [1]:
# For handling the timeseries
import pandas as pd, os, datetime
import numpy as np

# Statistical analysis
from sklearn.cluster import KMeans
from sklearn import metrics
from sklearn.metrics import silhouette_samples, silhouette_score

# Time series
from tslearn.clustering import TimeSeriesKMeans
from sktime.distances import dtw_distance
import skfuzzy as fuzz
from dtw import *

# For plotting
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import plotly.graph_objects as go
import plotly.io as pio
import plotly.express as px
import matplotlib.lines as mlines
import cartopy.feature as cfeature
import cartopy.crs as ccrs
import seaborn as sns
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
pio.renderers.default = 'notebook'

# Import the loader function
os.chdir('/g/data/ng72/ms5578/ID_HW_BARRA')
from process_code import load_generation_data

/g/data/xp65/public/apps/med_conda/envs/analysis3-25.10/lib/python3.11/site-packages/torch/cuda/__init__.py:58: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
2025-11-13 15:52:29,422 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 12GB due to system memory limit of 9.00 GiB
2025-11-13 15:52:29,425 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 12GB due to system memory limit of 9.00 GiB
2025-11-13 15:52:29,427 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 12GB due to system memory limit of 9.00 GiB
2025-11-13 15:52:29,429 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 12GB due to system memory limit of 9.00 GiB
2025-11-13 15:52:29,432 - distributed.nanny.memory - WARNING - Ignoring provided memor

Dask dashboard: /proxy/8787/status


In [2]:
df, info = load_generation_data(
    sdate="2009-07-01",
    edate="2024-06-30",
    mode="hourly",
    ftype=["Wind"],
    apply_remove_negatives=True,
    apply_remove_wind_zeros=True,
    wind_zero_threshold=40,
    apply_min_heatwave_days=True,
    min_heatwave_days_threshold=20,
    apply_clear_agc=False
)

Read gen_details & hw_tseries with Dask: 0.04 sec
Select group: 0.00 sec
--- Starting Dask-Native Process ---


/g/data/ng72/ms5578/ID_HW_BARRA/process_code.py:168: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



Starting final Dask compute...
Dask compute finished.

--- DASK TIMING REPORT ---
duid_setup: 0.00 seconds
csv_bulk_read: 0.20 seconds
filter_and_clean: 0.01 seconds
type_conversion_and_dropna: 0.02 seconds
date_filter: 0.01 seconds
hw_tseries_filter: 12.78 seconds
aggregate_hourly: 0.03 seconds
jitter: 0.21 seconds
final_merge: 0.02 seconds
compute: 66.90 seconds
--- END REPORT ---

Process group: 85.56 sec

--- DEBUG: Calculated Heatwave Days per Generator ---
DUID
MUSSELR1    334
SAPHWF1     206
WRWF1       201
HALLWF2     187
HALLWF1     187
GUNNING1    178
WOODLWN1    171
LKBONNY2    168
GULLRWF1    164
NBHWF1      162
WATERLWF    161
CLEMGPWF    158
LKBONNY3    157
SNOWTWN1    157
BLUFF1      151
TARALGA1    150
MEWF1       143
BOCORWF1    126
BODWF1      122
STWF1       115
SNOWNTH1    109
OAKLAND1    108
COOPGWF1    103
BALDHWF1    101
HDWF1        98
MACARTH1     98
HDWF3        92
HDWF2        92
SNOWSTH1     89
GRANWF1      84
MERCER01     84
CROOKWF2     76
ARWF1        72


In [3]:
client = Client(n_workers=5,
    threads_per_worker=1,
    memory_limit=f"{int(5)}GB"
)

client

/g/data/xp65/public/apps/med_conda/envs/analysis3_edge-25.11/lib/python3.11/site-packages/distributed/node.py:187: UserWarning:

Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 37075 instead



Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/37075/status,
Dashboard: /proxy/37075/status,Workers: 5
Total threads: 5,Total memory: 23.28 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:42775,Workers: 0
Dashboard: /proxy/37075/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:37855,Total threads: 1
Dashboard: /proxy/46291/status,Memory: 4.66 GiB
Nanny: tcp://127.0.0.1:34741,


In [4]:
# Single DUID test
gen = df[df['DUID']=='GUNNING1'].copy()

Using pandas

In [18]:
def pivot_day_tseries(df):
    # This function pivots the given df so it can be used with euclidean_distances. This includes removing the final (incomplete) timeseries.
    df['date'] = df['time'].dt.date
    df['hour'] = df['time'].dt.hour
    
    df_pivot = df.pivot_table(index='date', columns='hour', values='TOTALMWh').dropna()
    return df_pivot

hw_tseries = pivot_day_tseries(gen[gen['EHF_flag'] == 1].copy())
bl_tseries = pivot_day_tseries(gen[gen['EHF_flag'] == 0].copy())

Using NumPy

In [45]:
# Calculate 3D matrix of euclidean distances for each hw-baseline pair
hw_arr = hw_tseries.to_numpy()   # shape (m, 24)
bl_arr = bl_tseries.to_numpy()   # shape (n, 24)

diff = hw_arr[:, np.newaxis, :] - bl_arr[np.newaxis, :, :]

In [49]:
from sklearn.metrics.pairwise import euclidean_distances
distance_matrix = euclidean_distances(hw_arr, bl_arr)

In [52]:
from sklearn.cluster import KMeans

# Suppose hw_arr is (m, 24) - each row is a time series

# ---- Step 1: KMeans clustering ----
n_clusters = 4  # choose number of clusters
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
labels = kmeans.fit(distance_matrix)  # cluster assignment for each series

# cluster centers are the mean time series
cluster_centers = kmeans.cluster_centers_  # shape (n_clusters, 24)